# Layer 20: SIMD `EnzRef` Catalytic Role-Contrast Kinetics, Reaction SMILES Splitter & Disordered Microprotein `pSLiM` Tethers

This notebook demonstrates and verifies the **Layer 20 (`PR #19`)** primitives in `pg_bio`:
1. **Reaction SMILES Decomposition & Agent Stripping (`bio_split_reaction_roles`, `bio_reaction_role_vectors`)**
2. **Reference-Centered SIMD `EnzRef` Catalytic Role-Contrast (`bio_enzref_kinetics`, `bio_enzref_flux_rank`, `bio_enzref_kinetics_g128`, `bio_enzref_kinetics_smiles`)** with exact reaction-reversal antisymmetry (`FluxRank(e, p, r) == -FluxRank(e, r, p)`) and dead-binder elimination (`p == r -> 0.0`).
3. **Disordered Microprotein `pSLiM` Window Pooler (`bio_pslim_window_pool`) & Dual-SLiM Metabolon Tether (`bio_match_dual_slim_tether`, `bio_audit_fusion_junction_sanity`)** with in-memory corpus ZCA whitening, exact L2 unit normalization (`||w||_2 == 1.0`), and hydrophobic-core linker junction sanity gating.

In [1]:
import json
import numpy as np
import pandas as pd

# Load the synced Dual-Ledger Research Snapshot
with open('../data/research_ledger.json') as f:
    ledger = json.load(f)

df_attempts = pd.DataFrame(ledger['research_attempts'])[['attempt_code', 'run_date', 'category', 'status', 'pr_number', 'blog_post_slug']]
display(df_attempts)

,attempt_code,run_date,category,status,pr_number,blog_post_slug
0,ATT-2026-11-26-3DI-PROST-SKETCH,2026-11-26,PLM_Kinetics,Proposed,NaN,NaN
1,ATT-2026-11-26-ALLOSTERIC-ORTHO-COUPLING,2026-11-26,Molecular_Dynamics,Proposed,NaN,NaN
2,ATT-2026-11-26-IPSAE-MINIPAE-GATE,2026-11-26,Structural_Bio,Proposed,NaN,NaN
3,ATT-2026-11-26-PROSUPPORT-COMPOSITE-GIST,2026-11-26,Postgres_Systems,Proposed,NaN,NaN
4,ATT-2026-11-26-PG18-READSTREAM-G128,2026-11-26,Postgres_Systems,Proposed,NaN,NaN
5,ATT-2026-11-26-ENZREF-PSLIM,2026-11-26,PLM_Kinetics,Merged,19.0,pocketvina-posebusters-enzref-pslim-tethers-pg...
6,ATT-2026-11-24-POCKETVINA-POSEBUSTERS,2026-11-24,Structural_Bio,Merged,16.0,pocketvina-posebusters-enzref-pslim-tethers-pg...
7,ATT-2026-11-23-KNOW-THY-NEIGHBOUR-ZCA,2026-11-23,PLM_Kinetics,Published,15.0,know-thy-neighbour-zca-whitening-inline-biovec...
8,ATT-2026-11-21-LAYERED-CENTROIDS-FASTSCAN,2026-11-21,Postgres_Systems,Published,14.0,layered-centroids-register-fastscan-morton-oct...
9,ATT-2026-11-18-PRISM-RABITQ,2026-11-18,Postgres_Systems,Published,13.0,from-27-hours-to-3-minutes-tigerdata-pg-vector...


In [2]:
# Verify EnzRef Reference-Centered Algebraic Moments & Exact Antisymmetry
def enzref_moments_and_flux(enz_bg: np.ndarray, reactant: np.ndarray, product: np.ndarray):
    d_react = reactant - enz_bg
    d_prod = product - enz_bg
    c_fwd = d_prod - d_react
    m = 0.5 * (d_react + d_prod)
    norm_r = np.linalg.norm(d_react)
    norm_p = np.linalg.norm(d_prod)
    norm_c = np.linalg.norm(c_fwd)
    norm_m = np.linalg.norm(m)
    if norm_c < 1e-8 or (norm_r < 1e-8 and norm_p < 1e-8):
        return 0.0, 0.0, 0.0
    cos_m_c = 0.5 * (norm_p**2 - norm_r**2) / max(norm_m * norm_c, 1e-8)
    eps = 1e-6
    scale_amp = np.log(1.0 + (norm_m * norm_c) / np.sqrt(len(enz_bg)))
    flux = (0.72 * cos_m_c + 0.28 * np.tanh(np.log((norm_p + eps) / (norm_r + eps)))) * scale_amp
    return float(flux), float(norm_c), float(norm_m)

e = np.array([1.0, 0.0, 0.0, 0.0, 0.5, -0.5, 0.2, 0.1], dtype=np.float32)
r = np.array([2.0, 1.0, 0.0, 0.0, 0.5, -0.5, 0.2, 0.1], dtype=np.float32)
p_fwd = np.array([3.5, 2.5, 0.0, 0.0, 0.5, -0.5, 0.2, 0.1], dtype=np.float32)

flux_fwd, nc_fwd, nm_fwd = enzref_moments_and_flux(e, r, p_fwd)
flux_rev, nc_rev, nm_rev = enzref_moments_and_flux(e, p_fwd, r)
flux_dead, _, _ = enzref_moments_and_flux(e, r, r)

summary_df = pd.DataFrame([
    {'Mode': 'Forward Catalysis (r -> p)', 'FluxRank': round(flux_fwd, 6), 'RoleContrastNorm': round(nc_fwd, 4), 'MeanCatalyticResponse': round(nm_fwd, 4)},
    {'Mode': 'Reversed Reaction (p -> r)', 'FluxRank': round(flux_rev, 6), 'RoleContrastNorm': round(nc_rev, 4), 'MeanCatalyticResponse': round(nm_rev, 4)},
    {'Mode': 'Dead Non-Catalytic Binder (p == r)', 'FluxRank': round(flux_dead, 6), 'RoleContrastNorm': 0.0, 'MeanCatalyticResponse': round(np.linalg.norm(r - e), 4)},
])
assert abs(flux_fwd + flux_rev) < 1e-6, 'Exact antisymmetry violated!'
assert flux_dead == 0.0, 'Dead binder must score 0.0!'
display(summary_df)

,Mode,FluxRank,RoleContrastNorm,MeanCatalyticResponse
0,Forward Catalysis (r -> p),0.968413,2.1213,2.4749
1,Reversed Reaction (p -> r),-0.968413,2.1213,2.4749
2,Dead Non-Catalytic Binder (p == r),0.000000,0.0000,1.4142
